In [6]:
# @title 挂载google driver
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
# @title 安装依赖
!pip install -q git+https://github.com/Disty0/sdnq git+https://github.com/huggingface/diffusers
!pip install -q transformers accelerate safetensors psutil huggingface_hub

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
# @title INT8 Per-Channel 量化 (自动格式转换 + 低内存合并版)
import os
import json
import gc
import shutil
import torch
import psutil
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output
from safetensors.torch import save_file, load_file
from huggingface_hub import hf_hub_download, HfApi

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# ==========================================
# 1. 基础工具
# ==========================================
def print_mem():
    mem = psutil.virtual_memory()
    used_gb = mem.used / 1e9
    total_gb = mem.total / 1e9
    percent = mem.percent
    print(f"💻 RAM: {used_gb:.1f}GB / {total_gb:.1f}GB ({percent:.1f}%)")
    if percent > 85:
        print("⚠️ 内存使用超过 85%，强制 GC")
        gc.collect()

def force_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

target_modules_to_not_convert = [
    "layers.0.adaLN_modulation.0.weight",
    "all_x_embedder",
    "t_embedder",
    "cap_embedder",
    "all_final_layer"
]

def should_quantize(name):
    if name in target_modules_to_not_convert:
        return False
    for skip_prefix in target_modules_to_not_convert:
        if name.startswith(skip_prefix):
            return False
    return name.endswith(".weight")

def quantize_tensor_int8_per_channel(tensor):
    if tensor.dtype == torch.int8:
        return tensor, None
    t_float = tensor.float()
    if tensor.dim() == 2:
        abs_max = t_float.abs().amax(dim=1, keepdim=True).clamp(min=1e-8)
        scale   = abs_max / 127.0
        q       = (t_float / scale).round().clamp(-128, 127).to(torch.int8)
        return q, scale.to(torch.bfloat16)
    else:
        abs_max = t_float.abs().max().item()
        if abs_max < 1e-8:
            return tensor.to(torch.int8), torch.tensor(1.0, dtype=torch.bfloat16)
        scale = abs_max / 127.0
        q     = (t_float / scale).round().clamp(-128, 127).to(torch.int8)
        return q, torch.tensor(scale, dtype=torch.bfloat16)

# ==========================================
# 2. 格式检测（轻量，只读键名）
# ==========================================
def detect_format(shard_path):
    """只读键名，不加载张量，判断是否需要格式转换。"""
    data   = load_file(shard_path)
    keys   = list(data.keys())
    del data
    force_cleanup()

    sample = keys[:100]
    needs  = (
        any(k.startswith("model.diffusion_model.") for k in sample) or
        any("attention.qkv"    in k for k in sample) or
        any("attention.out.weight" in k for k in sample) or
        any("attention.q_norm" in k for k in sample) or
        any("final_layer."     in k for k in sample)
    )
    return needs

# ==========================================
# 3. 格式转换生成器（逐键 yield，零中间文件）
# ==========================================
def iter_converted_params(shard_path, needs_conversion):
    """
    逐键加载并转换，以生成器方式 yield (new_key, tensor)。
    不在内存中积累完整字典，不写中间文件。
    转换规则（ComfyUI → diffusers）：
      1. 去除 model.diffusion_model. 前缀
      2. final_layer.*         → all_final_layer.2-1.*
      3. x_embedder.*          → all_x_embedder.2-1.*
      4. attention.qkv.weight  → to_q / to_k / to_v（三等分）
      5. attention.out.weight  → attention.to_out.0.weight
      6. attention.q_norm      → attention.norm_q
      7. attention.k_norm      → attention.norm_k
    """
    data = load_file(shard_path)
    for k in list(data.keys()):
        v = data[k].cpu()
        del data[k]   # 逐键释放原始引用

        if not needs_conversion:
            yield k, v
            del v
            continue

        # ── 1. 去除前缀 ──────────────────────────────────────
        new_k = k.replace("model.diffusion_model.", "")

        # ── 2. final_layer → all_final_layer.2-1 ─────────────
        new_k = new_k.replace("final_layer.", "all_final_layer.2-1.")

        # ── 3. x_embedder → all_x_embedder.2-1 ──────────────
        new_k = new_k.replace("x_embedder.", "all_x_embedder.2-1.")

        # ── 4. attention.qkv.weight → to_q / to_k / to_v ────
        if new_k.endswith("attention.qkv.weight"):
            if v.shape[0] % 3 != 0:
                print(f"   ⚠️  {k} shape={list(v.shape)} 无法三等分，跳过")
                del v
                continue
            base = new_k[: new_k.rfind("attention.qkv.weight")]
            dim  = v.shape[0] // 3
            yield f"{base}attention.to_q.weight", v[:dim].clone()
            yield f"{base}attention.to_k.weight", v[dim:2*dim].clone()
            yield f"{base}attention.to_v.weight", v[2*dim:].clone()
            del v
            continue

        # ── 5. attention.out.weight → to_out.0.weight ────────
        new_k = new_k.replace("attention.out.weight",
                               "attention.to_out.0.weight")

        # ── 6. q_norm / k_norm → norm_q / norm_k ─────────────
        new_k = new_k.replace("attention.q_norm.weight",
                               "attention.norm_q.weight")
        new_k = new_k.replace("attention.k_norm.weight",
                               "attention.norm_k.weight")

        yield new_k, v
        del v

    # data 此时应已全空，但仍显式释放
    del data
    force_cleanup()

# ==========================================
# 4. 下载工具
# ==========================================
def download_from_url(url, dest_dir, progress_bar=None, progress_label=None):
    headers = {"User-Agent": "Mozilla/5.0"}
    with requests.get(url, stream=True, headers=headers, timeout=60) as r:
        r.raise_for_status()
        filename = None
        cd = r.headers.get("Content-Disposition", "")
        if "filename=" in cd:
            filename = cd.split("filename=")[-1].strip().strip('"').strip("'")
        if not filename:
            filename = url.split("?")[0].rstrip("/").split("/")[-1]
        if not filename or "." not in filename:
            filename = "model.safetensors"

        local_path = os.path.join(dest_dir, filename)
        total      = int(r.headers.get("content-length", 0))
        downloaded = 0
        chunk_size = 8 * 1024 * 1024

        if progress_bar is not None:
            progress_bar.max   = total if total > 0 else 1
            progress_bar.value = 0
        if progress_label is not None:
            progress_label.value = f"📄 {filename}  |  总大小: {total/1e9:.2f} GB" if total else f"📄 {filename}"

        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if progress_bar is not None and total > 0:
                        progress_bar.value = downloaded
                    if progress_label is not None:
                        pct = downloaded / total * 100 if total else 0
                        progress_label.value = (
                            f"⬇️  {downloaded/1e9:.2f}/{total/1e9:.2f} GB "
                            f"({pct:.1f}%) — {filename}"
                        )
        if progress_bar is not None and total > 0:
            progress_bar.value = total
        if progress_label is not None:
            progress_label.value = f"✅ 下载完成: {filename}  ({downloaded/1e9:.2f} GB)"
    return local_path, filename

# ==========================================
# 5. 配置工具
# ==========================================
def fetch_base_config(repo_id, subfolder, temp_dir):
    base_config = {}
    config_path = f"{subfolder}/config.json" if subfolder else "config.json"
    try:
        cf = hf_hub_download(repo_id, config_path,
                              cache_dir=temp_dir, resume_download=True)
        with open(cf) as f:
            base_config = json.load(f)
        base_config.pop("siglip_feat_dim", None)
        print(f"   ✓ 成功从 {repo_id} 获取 config.json")
    except Exception:
        print(f"   ⚠️ 在 {repo_id} 中未找到 config.json，尝试官方备用...")
        try:
            cf = hf_hub_download("Tongyi-MAI/Z-Image",
                                  "transformer/config.json",
                                  cache_dir=temp_dir, resume_download=True)
            with open(cf) as f:
                base_config = json.load(f)
            base_config.pop("siglip_feat_dim", None)
            print("   ✓ 成功借用官方版 config.json")
        except Exception as e:
            print(f"   ❌ 获取配置失败，将使用空配置: {e}")
    return base_config

def _write_configs(local_dir, base_config):
    print("\n📝 写入配置文件...")
    quant_cfg = {
        "add_skip_keys": False, "dequantize_fp32": False, "group_size": -1,
        "is_integer": True, "is_training": False, "modules_dtype_dict": {},
        "modules_to_not_convert": target_modules_to_not_convert,
        "non_blocking": False, "quant_conv": False, "quant_method": "sdnq",
        "quantization_device": None, "quantized_matmul_dtype": None,
        "return_device": None, "sdnq_version": "0.1.2", "svd_rank": 32,
        "svd_steps": 8, "use_grad_ckpt": True, "use_quantized_matmul": False,
        "use_quantized_matmul_conv": False, "use_static_quantization": True,
        "use_stochastic_rounding": False, "use_svd": False, "weights_dtype": "int8"
    }
    with open(os.path.join(local_dir, "quantization_config.json"), "w") as f:
        json.dump(quant_cfg, f, indent=2)

    base_config["quantization_config"] = {
        **quant_cfg,
        "quantization_device": "xpu", "return_device": "cpu", "add_skip_keys": True
    }
    base_config["_diffusers_version"] = "0.36.0.dev0"
    with open(os.path.join(local_dir, "config.json"), "w") as f:
        json.dump(base_config, f, indent=2)

# ==========================================
# 6. 量化核心（转换 + 量化一体，零中间文件）
# ==========================================
def _run_quant_core(all_shard_paths, base_config, local_dir, temp_dir, final_dir,
                    quant_progress_bar=None, quant_progress_label=None):

    TEMP_SHARD_SIZE    = 500_000_000  # 500 MB
    temp_shard_idx     = 1
    current_temp_shard = {}
    current_temp_size  = 0
    scales_dict        = {}
    quantized_count    = 0
    kept_count         = 0
    total_shards       = len(all_shard_paths)

    if quant_progress_bar is not None:
        quant_progress_bar.max   = total_shards
        quant_progress_bar.value = 0

    # ── 格式检测（仅读键名，一次性完成）────────────────────────
    print(f"\n🔍 检测权重格式...")
    needs_conversion = detect_format(all_shard_paths[0])
    if needs_conversion:
        print("   🔧 检测到 ComfyUI/非官方格式，量化时同步转换（无中间文件）")
    else:
        print("   ✓ 标准 diffusers 格式，直接量化")
    print_mem()

    # ── 逐块：转换 + 量化 一体化 ─────────────────────────────
    print(f"\n🔄 开始量化 (共 {total_shards} 块)...")
    for shard_idx, shard_path in enumerate(all_shard_paths):
        shard_name = os.path.basename(shard_path)
        print(f"\n{'─'*50}\n📦 [{shard_idx+1}/{total_shards}] {shard_name}")
        if quant_progress_label is not None:
            quant_progress_label.value = (
                f"⚙️  处理 {shard_idx+1}/{total_shards}: {shard_name}"
            )
        print_mem()

        # 通过生成器逐键消费，内存中同时只有单个张量
        for param_name, tensor in iter_converted_params(shard_path, needs_conversion):
            tensor = tensor.cpu()

            if should_quantize(param_name) and tensor.dim() >= 2:
                q_tensor, scale = quantize_tensor_int8_per_channel(tensor)
                if scale is not None:
                    scales_dict[param_name.replace(".weight", ".scale")] = scale
                    quantized_count += 1
            else:
                q_tensor   = (tensor.to(torch.bfloat16)
                              if tensor.dtype != torch.bfloat16 else tensor)
                kept_count += 1

            param_size = q_tensor.numel() * q_tensor.element_size()

            # 达到临时分片阈值则落盘
            if current_temp_size + param_size > TEMP_SHARD_SIZE and current_temp_shard:
                tmp = os.path.join(temp_dir, f"temp_{temp_shard_idx:04d}.safetensors")
                print(f"   💾 暂存临时分片 #{temp_shard_idx}")
                save_file(current_temp_shard, tmp)
                current_temp_shard.clear()
                current_temp_size = 0
                temp_shard_idx   += 1
                force_cleanup()

            current_temp_shard[param_name] = q_tensor
            current_temp_size += param_size
            del tensor, q_tensor

        if quant_progress_bar is not None:
            quant_progress_bar.value = shard_idx + 1

    # 最后一批落盘
    if current_temp_shard:
        tmp = os.path.join(temp_dir, f"temp_{temp_shard_idx:04d}.safetensors")
        save_file(current_temp_shard, tmp)
        del current_temp_shard
        force_cleanup()
    total_temp_shards = temp_shard_idx

    if scales_dict:
        save_file(scales_dict,
                  os.path.join(temp_dir, "temp_scales.safetensors"))
        del scales_dict
        force_cleanup()

    print(f"\n✅ 量化完成: {quantized_count} 层已量化 / {kept_count} 层保留")
    if quant_progress_label is not None:
        quant_progress_label.value = (
            f"✅ 量化完成: {quantized_count} 层量化 / {kept_count} 层保留"
        )

    # ── 拼装最终权重 ─────────────────────────────────────────
    print("\n🔗 拼装最终权重...")
    final_state_dict = {}
    for i in range(1, total_temp_shards + 1):
        tmp = os.path.join(temp_dir, f"temp_{i:04d}.safetensors")
        if os.path.exists(tmp):
            final_state_dict.update(load_file(tmp))
            force_cleanup()
    scales_file = os.path.join(temp_dir, "temp_scales.safetensors")
    if os.path.exists(scales_file):
        final_state_dict.update(load_file(scales_file))
        force_cleanup()

    print(f"💾 保存最终权重 ({len(final_state_dict)} 个参数)...")
    save_file(final_state_dict,
              os.path.join(local_dir, "diffusion_pytorch_model.safetensors"),
              metadata={"format": "pt"})
    del final_state_dict
    force_cleanup()

    _write_configs(local_dir, base_config)

    # ── 传输到 Drive ─────────────────────────────────────────
    print(f"\n📤 传输到 Google Drive: {final_dir}")
    os.makedirs(os.path.dirname(final_dir), exist_ok=True)
    if os.path.exists(final_dir):
        shutil.rmtree(final_dir)
    shutil.copytree(local_dir, final_dir)
    print("\n🎉 量化全部完成！文件已存入 GDrive。")
    print_mem()

# ==========================================
# 7. 目录常量
# ==========================================
FINAL_DIR = "/content/drive/MyDrive/z-image-base-transformer-int8"
LOCAL_DIR = "/content/quant_output"
TEMP_DIR  = "/content/temp_quant"

# ==========================================
# 8. UI 组件
# ==========================================
style        = {'description_width': 'initial'}
bar_layout   = widgets.Layout(width='680px', height='22px')
label_layout = widgets.Layout(width='680px')

mode_toggle = widgets.RadioButtons(
    options=['🤗 Repo ID 模式', '🌐 URL 直链模式'],
    value='🤗 Repo ID 模式',
    description='下载模式:',
    style=style,
    layout=widgets.Layout(width='400px')
)

repo_input     = widgets.Text(value='Tongyi-MAI/Z-Image',
                               description='Repo ID 或 HF URL:',
                               style=style, layout=widgets.Layout(width='500px'))
fetch_btn      = widgets.Button(description='🔍 获取文件列表', button_style='info')
file_dropdown  = widgets.Dropdown(description='选择权重文件:',
                                   style=style, layout=widgets.Layout(width='600px'))
start_repo_btn = widgets.Button(description='🚀 开始量化',
                                 button_style='success', disabled=True)
repo_dl_bar    = widgets.IntProgress(value=0, min=0, max=100,
                                      bar_style='info', layout=bar_layout)
repo_dl_label  = widgets.Label(value='', layout=label_layout)
repo_qt_bar    = widgets.IntProgress(value=0, min=0, max=100,
                                      bar_style='success', layout=bar_layout)
repo_qt_label  = widgets.Label(value='', layout=label_layout)

repo_panel = widgets.VBox([
    widgets.HBox([repo_input, fetch_btn]),
    widgets.HTML("<small><i>* 例如：Tongyi-MAI/Z-Image 或完整 HF URL</i></small>"),
    widgets.HBox([file_dropdown, start_repo_btn]),
    widgets.HTML("<b>下载进度:</b>"),
    repo_dl_bar, repo_dl_label,
    widgets.HTML("<b>量化进度:</b>"),
    repo_qt_bar, repo_qt_label,
])

url_input              = widgets.Text(value='', description='文件直链 URL:',
                                       placeholder='https://civitai.com/... 或任意直链',
                                       style=style, layout=widgets.Layout(width='700px'))
config_repo_input      = widgets.Text(value='Tongyi-MAI/Z-Image-Turbo',
                                       description='借用配置的 Repo ID:',
                                       style=style, layout=widgets.Layout(width='500px'))
config_subfolder_input = widgets.Text(value='transformer',
                                       description='Config 子目录:',
                                       style=style, layout=widgets.Layout(width='400px'))
start_url_btn  = widgets.Button(description='🚀 开始 URL 量化', button_style='warning')
url_dl_bar     = widgets.IntProgress(value=0, min=0, max=100,
                                      bar_style='info', layout=bar_layout)
url_dl_label   = widgets.Label(value='', layout=label_layout)
url_qt_bar     = widgets.IntProgress(value=0, min=0, max=100,
                                      bar_style='success', layout=bar_layout)
url_qt_label   = widgets.Label(value='', layout=label_layout)

url_panel = widgets.VBox([
    url_input,
    widgets.HTML("<small><i>* 支持 Civitai、HuggingFace 等任意 .safetensors 直链</i></small>"),
    widgets.HBox([config_repo_input, config_subfolder_input]),
    widgets.HTML("<small><i>* 借用配置 Repo 与子目录；留空则使用官方备用配置</i></small>"),
    start_url_btn,
    widgets.HTML("<b>下载进度:</b>"),
    url_dl_bar, url_dl_label,
    widgets.HTML("<b>量化进度:</b>"),
    url_qt_bar, url_qt_label,
])

output_area = widgets.Output()

def on_mode_change(change):
    if change['new'] == '🤗 Repo ID 模式':
        repo_panel.layout.display = ''
        url_panel.layout.display  = 'none'
    else:
        repo_panel.layout.display = 'none'
        url_panel.layout.display  = ''

mode_toggle.observe(on_mode_change, names='value')
url_panel.layout.display = 'none'

# ==========================================
# 9. 回调函数
# ==========================================
def parse_repo_id(input_str):
    input_str = input_str.strip()
    if "huggingface.co/" in input_str:
        parts = input_str.split("huggingface.co/")[-1].split("/")
        if len(parts) >= 2:
            return f"{parts[0]}/{parts[1]}"
    return input_str

def on_fetch_clicked(b):
    with output_area:
        clear_output()
        repo_id = parse_repo_id(repo_input.value)
        print(f"🔄 正在获取 {repo_id} 的文件列表...")
        try:
            api   = HfApi()
            files = api.list_repo_files(repo_id)
            valid = [f for f in files
                     if f.endswith('.index.json') or f.endswith('.safetensors')]
            if not valid:
                print("❌ 未找到 .index.json 或 .safetensors 文件！")
                return
            file_dropdown.options  = valid
            default = "transformer/diffusion_pytorch_model.safetensors.index.json"
            file_dropdown.value    = default if default in valid else valid[0]
            start_repo_btn.disabled = False
            print("✅ 获取成功！请选择权重文件，然后点击【开始量化】。")
        except Exception as e:
            print(f"❌ 获取失败: {e}")

def run_repo_quantization(b):
    with output_area:
        clear_output()
        repo_id       = parse_repo_id(repo_input.value)
        selected_file = file_dropdown.value
        subfolder     = os.path.dirname(selected_file)

        print("="*60)
        print("🚀 开始 Per-Channel 量化流程 [Repo ID 模式]")
        print(f"📦 {repo_id} -> {selected_file}")
        print("="*60)
        print_mem()

        for d in [LOCAL_DIR, TEMP_DIR]:
            shutil.rmtree(d, ignore_errors=True)
            os.makedirs(d, exist_ok=True)
        repo_dl_bar.value = 0; repo_dl_label.value = ''
        repo_qt_bar.value = 0; repo_qt_label.value = ''

        print("\n📥 下载权重分片...")
        is_sharded = selected_file.endswith(".index.json")
        if is_sharded:
            index_path = hf_hub_download(repo_id, selected_file,
                                          cache_dir=TEMP_DIR, resume_download=True)
            with open(index_path) as f:
                weight_map = json.load(f)["weight_map"]
            shard_names = sorted(set(weight_map.values()))
            print(f"   ✓ 分片模型，共 {len(shard_names)} 片")
        else:
            shard_names = [os.path.basename(selected_file)]
            print("   ✓ 单文件模型")

        repo_dl_bar.max   = len(shard_names)
        repo_dl_bar.value = 0
        local_shard_paths = []
        for idx, shard_name in enumerate(shard_names):
            hf_path = f"{subfolder}/{shard_name}" if subfolder else shard_name
            repo_dl_label.value = f"⬇️  {idx+1}/{len(shard_names)}: {shard_name}"
            p = hf_hub_download(repo_id, hf_path,
                                 cache_dir=TEMP_DIR, resume_download=True)
            local_shard_paths.append(p)
            repo_dl_bar.value = idx + 1
        repo_dl_label.value = f"✅ 全部 {len(shard_names)} 片下载完成"

        print("\n📋 获取配置文件...")
        base_config = fetch_base_config(repo_id, subfolder, TEMP_DIR)

        _run_quant_core(local_shard_paths, base_config,
                        LOCAL_DIR, TEMP_DIR, FINAL_DIR,
                        repo_qt_bar, repo_qt_label)

def run_url_quantization(b):
    with output_area:
        clear_output()
        url              = url_input.value.strip()
        config_repo      = parse_repo_id(config_repo_input.value.strip())
        config_subfolder = config_subfolder_input.value.strip()

        if not url:
            print("❌ 请先填写文件直链 URL！")
            return

        print("="*60)
        print("🚀 开始 Per-Channel 量化流程 [URL 直链模式]")
        print(f"🌐 URL: {url}")
        print(f"📋 借用配置: {config_repo} / {config_subfolder or '(根目录)'}")
        print("="*60)
        print_mem()

        for d in [LOCAL_DIR, TEMP_DIR]:
            shutil.rmtree(d, ignore_errors=True)
            os.makedirs(d, exist_ok=True)
        url_dl_bar.value = 0; url_dl_label.value = ''
        url_qt_bar.value = 0; url_qt_label.value = ''

        print("\n📥 从 URL 下载权重文件...")
        try:
            local_shard_path, filename = download_from_url(
                url, TEMP_DIR,
                progress_bar=url_dl_bar,
                progress_label=url_dl_label
            )
        except Exception as e:
            print(f"❌ 下载失败: {e}")
            return

        print("\n📋 获取配置文件...")
        base_config = fetch_base_config(config_repo, config_subfolder, TEMP_DIR)

        _run_quant_core([local_shard_path], base_config,
                        LOCAL_DIR, TEMP_DIR, FINAL_DIR,
                        url_qt_bar, url_qt_label)

fetch_btn.on_click(on_fetch_clicked)
start_repo_btn.on_click(run_repo_quantization)
start_url_btn.on_click(run_url_quantization)

# ==========================================
# 10. 渲染 UI
# ==========================================
ui_box = widgets.VBox([
    widgets.HTML("<h3>🎛️ Z-Image INT8 量化面板 (自动格式转换 + 低内存版)</h3>"),
    mode_toggle,
    widgets.HTML("<hr>"),
    repo_panel,
    url_panel,
    widgets.HTML("<hr>"),
    output_area
])

print("✅ UI 加载完毕。")
display(ui_box)

✅ UI 加载完毕。


In [7]:
import shutil
import os
import errno

source_dir = "/content/drive1/Drive/z-image-base-transformer-int8"
target_dir = "/content/drive/MyDrive/z-image-base-transformer-int8"

# 自定义拷贝函数，兼容旧版 Python
def copy_directory(src, dst):
    try:
        shutil.copytree(src, dst)
    except OSError as e:
        # 如果目标文件夹已存在，跳过
        if e.errno == errno.EEXIST:
            print(f"目标文件夹已存在，跳过拷贝: {dst}")
        else:
            raise e

if os.path.exists(source_dir):
    try:
        copy_directory(source_dir, target_dir)
        print("文件夹拷贝完成！")
    except Exception as e:
        print(f"拷贝失败: {e}")
else:
    print(f"源文件夹不存在: {source_dir}")

文件夹拷贝完成！
